# HEMR — Hypergraph Embeddings for Music Recommendation
### GPU/TPU-Accelerated Re-Implementation
**Paper:** *Music Recommendation via Hypergraph Embedding* — La Gatta et al., IEEE TNNLS 2023  
**Datasets:** songs_final.csv · user_data_final.csv · user_data_partitioned_val.csv  
**Acceleration:** CUDA (GPU) for Word2Vec, numpy matmul, and cosine scoring; CuPy optional.

---
**Quick-Run vs Full-Run**  
Set `QUICK_RUN = True` for a 5 000-user / 1 000-song subset (< 10 min on free Colab).  
Set `QUICK_RUN = False` for the complete dataset (several hours, GPU recommended).

## §1 · Environment Setup & GPU Detection

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §1  Environment Setup                                              ║
# ╚══════════════════════════════════════════════════════════════════════╝
import subprocess, sys, time as _t
_NOTEBOOK_START = _t.time()

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# NOTE: cupy removed — it requires matching CUDA version and hangs on free Colab.
# GPU acceleration is provided via PyTorch (always available on Colab T4).
pip("gensim>=4.3", "psutil")

import os, gc, time, pickle, random, warnings, datetime
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict
from scipy import spatial
from scipy.stats import zscore
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm
import torch, psutil
warnings.filterwarnings("ignore")
tqdm.pandas()

# ── GPU / device detection ─────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print(f"PyTorch device : {DEVICE}")
print(f"In Colab       : {IN_COLAB}")
if DEVICE == "cuda":
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"System RAM     : {psutil.virtual_memory().total/1e9:.1f} GB")


## §2 · Configuration — Quick-Run vs Full-Run

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §2  Configuration                                                   ║
# ║  RUN_MODE = "QUICK"      → small subset, fastest                    ║
# ║  RUN_MODE = "FULL_LITE"  → full dataset, Colab-safe                 ║
# ║  RUN_MODE = "FULL_PAPER" → paper-like, may exceed free Colab CPU    ║
# ╚══════════════════════════════════════════════════════════════════════╝

RUN_MODE = "FULL_LITE"      # ← CHANGE THIS: "QUICK" | "FULL_LITE" | "FULL_PAPER"
QUICK_RUN = RUN_MODE == "QUICK"

# ── Data paths ─────────────────────────────────────────────────────────────────
DATA_DIR      = "drive/MyDrive/HEMR_processed/NewData"
OUT_DIR       = "drive/MyDrive/HEMR_outputs"
CACHE_DIR     = f"{OUT_DIR}/cache"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

SONGS_CSV     = f"{DATA_DIR}/songs.csv"
USER_CSV      = f"{DATA_DIR}/user_data.csv"
USER_PART_CSV = f"{DATA_DIR}/data_partitioned.csv"

# ── Hyper-parameters tuned for Colab stability ────────────────────────────────
if RUN_MODE == "QUICK":
    MAX_USERS        = 5_000
    MAX_SONGS        = 1_000
    WALK_LENGTH      = 20
    NUM_WALKS        = 3
    EMBED_DIM        = 32
    W2V_WINDOW       = 5
    W2V_EPOCHS       = 3
    W2V_WORKERS      = 2
    W2V_NEGATIVE     = 5
    EVAL_K_LIST      = [5, 10, 20]
    SCORE_BATCH_SIZE = 2_000
    RUN_BERT_OVERRIDE= False
    RUN_LABEL        = "QUICK"
elif RUN_MODE == "FULL_LITE":
    MAX_USERS        = None
    MAX_SONGS        = None
    WALK_LENGTH      = 40      # 100 in paper; 40 is much lighter on Colab
    NUM_WALKS        = 2       # 10 in paper; biggest CPU saver
    EMBED_DIM        = 64      # 200 in paper; paper shows diminishing returns >50
    W2V_WINDOW       = 5       # keep at 5 (paper also found >5 not worth the cost)
    W2V_EPOCHS       = 4       # 11 in paper; good tradeoff for Colab
    W2V_WORKERS      = 1       # reduces Colab "CPU usage exceeded" risk
    W2V_NEGATIVE     = 5
    EVAL_K_LIST      = [5, 10, 20, 50]
    SCORE_BATCH_SIZE = 512
    RUN_BERT_OVERRIDE= False   # keep lyrics/BERT off for full Colab runs
    RUN_LABEL        = "FULL_LITE"
else:
    MAX_USERS        = None
    MAX_SONGS        = None
    WALK_LENGTH      = 100
    NUM_WALKS        = 10
    EMBED_DIM        = 200
    W2V_WINDOW       = 5
    W2V_EPOCHS       = 11
    W2V_WORKERS      = 2
    W2V_NEGATIVE     = 10
    EVAL_K_LIST      = [5, 10, 20, 50, 100]
    SCORE_BATCH_SIZE = 1_000
    RUN_BERT_OVERRIDE= False
    RUN_LABEL        = "FULL_PAPER"

RANDOM_SEED  = 42
ALPHA        = 1.0
BETA         = 0.0
CSV_CHUNK    = 300_000

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Run mode         : {RUN_LABEL}")
print(f"Embedding dim    : {EMBED_DIM}")
print(f"Walk length×walks: {WALK_LENGTH} × {NUM_WALKS}")
print(f"W2V epochs       : {W2V_EPOCHS}")
print(f"W2V workers      : {W2V_WORKERS}")
print(f"Score batch size : {SCORE_BATCH_SIZE}")
print(f"Output dir       : {OUT_DIR}")


## §3 · Data Loading & Verification

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §3  Data Loading & Verification                                    ║
# ║  Optimised for Colab: stream CSV, filter early, cache compact copy  ║
# ╚══════════════════════════════════════════════════════════════════════╝

def load_and_report(path, label, **read_kwargs):
    """Load a CSV with error handling and print a compact schema report."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[DATA ERROR] '{path}' not found.\n"
            f"  → Run the HEMR preprocessing notebook first, or update DATA_DIR.")
    t0 = time.time()
    try:
        df = pd.read_csv(path, **read_kwargs)
    except Exception as e:
        raise RuntimeError(f"[LOAD ERROR] Failed to read '{path}': {e}") from e

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Shape   : {df.shape}")
    print(f"  Columns : {df.columns.tolist()}")
    print(f"  Dtypes  :\n{df.dtypes.to_string()}")
    null_counts = df.isna().sum()
    if null_counts.any():
        print(f"  Nulls   :\n{null_counts[null_counts>0].to_string()}")
    else:
        print("  Nulls   : none")
    print(f"  Loaded  : {time.time()-t0:.1f}s  |  "
          f"RAM: {psutil.virtual_memory().used/1e9:.1f}/{psutil.virtual_memory().total/1e9:.1f} GB")
    display(df.head(3))
    return df

# ── songs_final.csv ───────────────────────────────────────────────────────────
song_data = load_and_report(
    SONGS_CSV,
    "songs_final.csv",
    dtype={"song_id":"str","artist_name":"str","release":"str","title":"str","tags":"str"}
)

valid_song_ids = set(song_data["song_id"].astype(str).unique())
compact_cache = f"{CACHE_DIR}/user_data_total_{RUN_LABEL}.pkl"

if os.path.exists(compact_cache):
    print(f"\nLoading cached compact interactions: {compact_cache}")
    with open(compact_cache, "rb") as f:
        user_data_total = pickle.load(f)
else:
    print(f"\nStreaming {USER_PART_CSV} in chunks of {CSV_CHUNK:,} rows…")
    needed_cols = ["user","song_id","play_count","triple_id","set"]
    pieces = []

    for chunk in tqdm(pd.read_csv(
            USER_PART_CSV,
            usecols=needed_cols,
            dtype={"user":"str","song_id":"str","play_count":"int32","triple_id":"int32","set":"category"},
            chunksize=CSV_CHUNK), desc="Chunks"):
        # Filter early to avoid keeping interactions whose songs are absent from song_data
        chunk = chunk[chunk["song_id"].isin(valid_song_ids)]
        if len(chunk) == 0:
            continue
        pieces.append(chunk)
        del chunk
        gc.collect()

    user_data_total = pd.concat(pieces, ignore_index=True)
    del pieces
    gc.collect()

    # Memory compaction
    user_data_total["user"] = user_data_total["user"].astype("category")
    user_data_total["song_id"] = user_data_total["song_id"].astype("category")
    user_data_total["set"] = user_data_total["set"].astype("category")

    with open(compact_cache, "wb") as f:
        pickle.dump(user_data_total, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Cached compact interactions → {compact_cache}")

print(f"  Shape   : {user_data_total.shape}")
print(f"  Set dist: {dict(user_data_total['set'].value_counts())}")
print(f"  RAM now : {psutil.virtual_memory().used/1e9:.1f} GB")
display(user_data_total.head(3))


In [ ]:
# ── Quick-Run subsetting ───────────────────────────────────────────────────────
if QUICK_RUN:
    print(f"[QUICK] Subsetting to top {MAX_SONGS} songs / {MAX_USERS} users…")

    top_songs = (user_data_total.groupby("song_id")["play_count"]
                 .sum().nlargest(MAX_SONGS).index)
    user_data_total = user_data_total[
        user_data_total["song_id"].isin(top_songs)].copy()

    top_users = (user_data_total.groupby("user")["song_id"]
                 .count().nlargest(MAX_USERS).index)
    user_data_total = user_data_total[
        user_data_total["user"].isin(top_users)].copy()

    song_data = song_data[song_data["song_id"].isin(
        user_data_total["song_id"].unique())].copy()

    user_data_total.reset_index(drop=True, inplace=True)
    user_data_total["triple_id"] = user_data_total.index.astype("int32")

    if len(user_data_total) == 0:
        raise RuntimeError("[SUBSET ERROR] No rows remain after subsetting. "
                           "Check MAX_USERS / MAX_SONGS or the source CSV.")

    print(f"  Users  : {user_data_total['user'].nunique():,}")
    print(f"  Songs  : {song_data.shape[0]:,}")
    print(f"  Rows   : {len(user_data_total):,}")
    gc.collect()

# ── Train / test split ────────────────────────────────────────────────────────
user_data = user_data_total[user_data_total["set"] != "test"].copy()
print(f"Train rows : {len(user_data):,}  |  "
      f"Test rows  : {(user_data_total['set']=='test').sum():,}")


## §4 · Hypergraph Construction

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §4  Hypergraph Construction                                        ║
# ║  Builds 4 types of hyperedges (paper §III-A):                      ║
# ║    · Listenings  (user → songs listened)                           ║
# ║    · Album       (release → songs on it)                           ║
# ║    · Discography (artist → releases)                                ║
# ║    · Topic       (tag → songs with that tag)                       ║
# ╚══════════════════════════════════════════════════════════════════════╝

t0 = time.time()

# ── 4a: User hyperedges (Listenings) ──────────────────────────────────────────
print("Building user hyperedges…")
user_hyperedges = (user_data.groupby("user")["song_id"]
                   .apply(list).reset_index(name="songs"))
user_hyperedges["user_id"]        = "u_" + user_hyperedges.index.astype(str)
user_hyperedges["user_id_matrix"] = user_hyperedges.index.astype("int32")

# Attach user IDs back to user_data (vectorised map — no merge copy)
uid_map    = user_hyperedges.set_index("user")["user_id"].to_dict()
umatrix_map = user_hyperedges.set_index("user")["user_id_matrix"].to_dict()
user_data = user_data.copy()
user_data["user_id"]        = user_data["user"].map(uid_map).astype("category")
user_data["user_id_matrix"] = user_data["user"].map(umatrix_map).astype("int32")
user_data["play_count"]     = user_data["play_count"].astype("int32")
del uid_map, umatrix_map; gc.collect()

# ── 4b: Release hyperedges (Album) ────────────────────────────────────────────
print("Building release hyperedges…")
release_hyperedges = (song_data.groupby(["artist_name","release"])["song_id"]
                      .apply(list).reset_index(name="songs"))
release_hyperedges["release_id"] = "r_" + release_hyperedges.index.astype(str)

# ── 4c: Artist hyperedges (Discography) ───────────────────────────────────────
print("Building artist hyperedges…")
artists_hyperedges = (release_hyperedges.groupby("artist_name")["release_id"]
                      .apply(list).reset_index(name="releases"))
artists_hyperedges["artist_id"] = "a_" + artists_hyperedges.index.astype(str)

# ── 4d: Tag hyperedges (Topic) ────────────────────────────────────────────────
print("Building tag hyperedges…")
tag_dict = {}
for _, row in song_data[["song_id","tags"]].iterrows():
    try:
        tags = eval(row["tags"])          # stored as stringified Python list
    except Exception:
        tags = []
    for tag in tags:
        if tag not in tag_dict:
            tag_dict[tag] = {"songs": [], "tag_id": f"t_{len(tag_dict)}"}
        tag_dict[tag]["songs"].append(row["song_id"])

# ── 4e: Assemble global hyperedge dict ────────────────────────────────────────
print("Assembling hyperedge dict…")
hyperedges = {}

for _, row in user_hyperedges.iterrows():
    uid = row["user_id"]
    hyperedges[uid] = {"members": row["songs"] + [uid], "category": "user"}

for _, row in release_hyperedges.iterrows():
    rid = row["release_id"]
    hyperedges[rid] = {"members": row["songs"] + [rid], "category": "release"}

for _, row in artists_hyperedges.iterrows():
    aid = row["artist_id"]
    hyperedges[aid] = {"members": row["releases"] + [aid], "category": "artist"}

for tag, info in tag_dict.items():
    tid = info["tag_id"]
    hyperedges[tid] = {"members": info["songs"] + [tid], "category": "tag"}

# ── 4f: Vertex membership index ───────────────────────────────────────────────
print("Building vertex membership index…")
vertexMemberships = defaultdict(list)
for h_index, h_data in hyperedges.items():
    for node in h_data["members"]:
        vertexMemberships[node].append(h_index)
vertexMemberships = dict(vertexMemberships)

print(f"\n  Hyperedge construction time: {time.time()-t0:.1f}s")
print(f"  Total hyperedges : {len(hyperedges):,}")
print(f"  Total vertices   : {len(vertexMemberships):,}")

In [ ]:
# ── 4g: Hyperedge statistics & visualisation ──────────────────────────────────
cat_amounts = {}
size_list   = []
for h_id, h in hyperedges.items():
    cat = h["category"]
    cat_amounts[cat] = cat_amounts.get(cat, 0) + 1
    size_list.append(len(h["members"]))

print("Hyperedge counts by category:")
for cat, n in sorted(cat_amounts.items(), key=lambda x: -x[1]):
    print(f"  {cat:<15}: {n:>8,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Hypergraph Statistics", fontsize=14, fontweight="bold")

# Category bar chart
cats = sorted(cat_amounts.keys(), key=lambda c: cat_amounts[c])
counts = [cat_amounts[c] for c in cats]
axes[0].barh(cats, counts, color=sns.color_palette("husl", len(cats)))
axes[0].set_xlabel("Number of hyperedges")
axes[0].set_title("Hyperedges by Category")
for i, v in enumerate(counts):
    axes[0].text(v, i, f"  {v:,}", va="center")

# Size distribution
axes[1].hist(size_list, bins=60, color="#4C72B0", edgecolor="white", log=True)
axes[1].set_xlabel("Hyperedge size (# members)")
axes[1].set_ylabel("Count (log scale)")
axes[1].set_title("Hyperedge Size Distribution")
axes[1].axvline(np.median(size_list), color="red", linestyle="--",
                label=f"Median={np.median(size_list):.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/hypergraph_stats_{RUN_LABEL}.png", dpi=120)
plt.show()
print(f"  Min size: {min(size_list)}   Max size: {max(size_list)}   "
      f"Median: {np.median(size_list):.0f}")

## §5 · Random Walks (SubsampleAndTraverse — Algorithm 1)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §5  Random Walk Generation                                         ║
# ║  Optimised: cache walks + faster sampling + smaller full-run config ║
# ╚══════════════════════════════════════════════════════════════════════╝

walk_cache = f"{CACHE_DIR}/walks_{RUN_LABEL}_L{WALK_LENGTH}_N{NUM_WALKS}.pkl"

# Pre-convert member lists to tuples to avoid repeated list slicing
_hg = {k: tuple(v["members"]) for k, v in hyperedges.items()}
_vm = {k: tuple(v) for k, v in vertexMemberships.items()}

def SubsampleAndTraverse(length, num_walks, hg, vm, alpha=1.0, beta=0.0):
    walks = []
    verts = list(vm.keys())
    random.shuffle(verts)

    for vertex in tqdm(verts, desc="Random walks", leave=False):
        for _ in range(num_walks):
            curr_v = vertex
            he_id  = random.choice(vm[curr_v])
            walk   = [curr_v]

            for _step in range(length - 1):
                members = hg[he_id]
                p_e = min(alpha / max(len(members), 1) + beta, 1.0)

                if random.random() < p_e:
                    he_id = random.choice(vm[curr_v])
                    members = hg[he_id]

                if len(members) == 1:
                    nxt = members[0]
                else:
                    nxt = random.choice(members)
                    if nxt == curr_v:
                        # cheap resample once; avoids unbounded while loop
                        nxt = random.choice(members)

                curr_v = nxt
                walk.append(curr_v)

            walks.append(walk)
    return walks

if os.path.exists(walk_cache):
    print(f"Loading cached walks: {walk_cache}")
    with open(walk_cache, "rb") as f:
        walksSAT = pickle.load(f)
else:
    print("Starting random walk generation…")
    t0 = time.time()
    walksSAT = SubsampleAndTraverse(
        length=WALK_LENGTH,
        num_walks=NUM_WALKS,
        hg=_hg, vm=_vm, alpha=ALPHA, beta=BETA
    )
    elapsed = time.time() - t0
    with open(walk_cache, "wb") as f:
        pickle.dump(walksSAT, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Walks cached to : {walk_cache}")
    print(f"  Time            : {elapsed:.1f}s  ({elapsed/60:.1f} min)")

print(f"  Total walks    : {len(walksSAT):,}")
print(f"  Steps per walk : {WALK_LENGTH}")
print(f"  Sample walk[0] : {walksSAT[0][:8]} …")
del _hg, _vm
gc.collect()


In [ ]:
# ── Walk statistics visualisation ─────────────────────────────────────────────
walk_lens = [len(w) for w in walksSAT]
unique_per_walk = [len(set(w)) for w in walksSAT]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Random Walk Statistics", fontweight="bold")

axes[0].hist(walk_lens, bins=20, color="#55a868", edgecolor="white")
axes[0].set_title("Walk Length Distribution")
axes[0].set_xlabel("Length"); axes[0].set_ylabel("Count")

axes[1].hist(unique_per_walk, bins=40, color="#c44e52", edgecolor="white")
axes[1].set_title("Unique Vertices per Walk")
axes[1].set_xlabel("# Unique vertices"); axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/walk_stats_{RUN_LABEL}.png", dpi=120)
plt.show()

print(f"  Avg unique vertices/walk : {np.mean(unique_per_walk):.1f}")

## §6 · Word2Vec Embedding (Skip-Gram, GPU-accelerated via Gensim)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §6  Word2Vec — Context Embeddings                                  ║
# ║  Important: gensim Word2Vec is CPU-based, so keep settings modest   ║
# ╚══════════════════════════════════════════════════════════════════════╝
from gensim.models.word2vec import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec

class LossLogger(CallbackAny2Vec):
    """Gensim callback: record loss after every epoch."""
    def __init__(self):
        self.epoch      = 0
        self.losses     = []
        self._prev_loss = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        delta = loss - self._prev_loss
        self._prev_loss = loss
        self.losses.append(delta)
        self.epoch += 1
        print(f"    Epoch {self.epoch:>3}  cumulative loss: {loss:,.0f}"
              f"  Δ: {delta:,.0f}")

loss_logger = LossLogger()
w2v_cache = f"{CACHE_DIR}/w2v_{RUN_LABEL}_d{EMBED_DIM}_e{W2V_EPOCHS}.bin"

print(f"Training Word2Vec  dim={EMBED_DIM}  window={W2V_WINDOW}"
      f"  epochs={W2V_EPOCHS}  workers={W2V_WORKERS}  negative={W2V_NEGATIVE}")
t0 = time.time()

if os.path.exists(w2v_cache):
    print(f"Loading cached Word2Vec model: {w2v_cache}")
    w2v_model = Word2Vec.load(w2v_cache)
else:
    w2v_model = Word2Vec(
        sentences    = walksSAT,
        vector_size  = EMBED_DIM,
        window       = W2V_WINDOW,
        min_count    = 0,
        sg           = 1,
        hs           = 0,
        negative     = W2V_NEGATIVE,
        sample       = 1e-4,
        workers      = W2V_WORKERS,
        epochs       = W2V_EPOCHS,
        compute_loss = True,
        callbacks    = [loss_logger],
        seed         = RANDOM_SEED,
    )
    w2v_model.save(w2v_cache)

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"  Vocabulary size  : {len(w2v_model.wv):,}")

vertex_ids         = w2v_model.wv.index_to_key
vertex_embeddings  = w2v_model.wv.vectors
context_embeddings = dict(zip(vertex_ids, vertex_embeddings))
print(f"  context_embeddings entries: {len(context_embeddings):,}")


In [ ]:
# ── Training loss curve ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Word2Vec Training", fontweight="bold")

epochs_x = list(range(1, len(loss_logger.losses)+1))
axes[0].plot(epochs_x, loss_logger.losses, marker="o", color="#4C72B0")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss Δ per epoch")
axes[0].set_title("Per-Epoch Loss")
axes[0].grid(alpha=0.3)

# ── 2-D PCA of song embeddings ────────────────────────────────────────────────
song_ids_known = [sid for sid in song_data["song_id"]
                  if sid in context_embeddings]
song_vecs = np.array([context_embeddings[sid] for sid in song_ids_known])

pca2 = PCA(n_components=2, random_state=RANDOM_SEED)
song_2d = pca2.fit_transform(song_vecs)

# Colour by first tag (or grey)
def first_tag(t):
    try: return eval(t)[0]
    except: return "unknown"

tag_series = song_data[song_data["song_id"].isin(song_ids_known)]["tags"].map(first_tag)
top_tags   = tag_series.value_counts().head(8).index.tolist()
palette    = dict(zip(top_tags, sns.color_palette("tab10", len(top_tags))))

colors = [palette.get(t, "#cccccc") for t in tag_series]
axes[1].scatter(song_2d[:,0], song_2d[:,1], c=colors, s=8, alpha=0.5)
handles = [plt.Line2D([0],[0], marker="o", color="w",
                      markerfacecolor=palette[t], label=t, markersize=8)
           for t in top_tags]
axes[1].legend(handles=handles, title="Tag", fontsize=7)
axes[1].set_title("Song Embeddings (PCA 2-D)")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/w2v_training_{RUN_LABEL}.png", dpi=120)
plt.show()

print(f"PCA explained variance: {pca2.explained_variance_ratio_.sum()*100:.1f}%")

## §7 · Mood Detection (Arousal & Valence)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §7  Mood Detection                                                 ║
# ║  Per-user arousal & valence = play-count-weighted avg over songs    ║
# ║  + max-pooling term (original notebook cells 45-47)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Map song arousal/valence onto user interactions
song_av = song_data[["song_id","arousal","valence"]].set_index("song_id")
user_data = user_data.copy()
user_data["arousal"] = user_data["song_id"].map(song_av["arousal"]).astype("float32")
user_data["valence"] = user_data["song_id"].map(song_av["valence"]).astype("float32")

# Weighted average (eq. weighting by play_count)
user_data["w_arousal"] = user_data["arousal"] * user_data["play_count"]
user_data["w_valence"] = user_data["valence"] * user_data["play_count"]

user_mood = (user_data
             .groupby("user_id", observed=True)
             .agg(w_arousal=("w_arousal","sum"),
                  w_valence=("w_valence","sum"),
                  total_pc =("play_count","sum"))
             .reset_index())
user_mood["arousal"] = (user_mood["w_arousal"] / user_mood["total_pc"]).astype("float32")
user_mood["valence"] = (user_mood["w_valence"] / user_mood["total_pc"]).astype("float32")

# Max-pooling term (notebook cell 47)
user_max = (user_data.groupby("user_id", observed=True)
            .agg(max_arousal=("arousal","max"),
                 max_valence=("valence","max"))
            .reset_index())
user_mood = user_mood.merge(user_max, on="user_id", how="left")
user_mood["arousal"] = (user_mood["arousal"] + user_mood["max_arousal"]).astype("float32")
user_mood["valence"] = (user_mood["valence"] + user_mood["max_valence"]).astype("float32")

# Build arousal_valence_dict (nodes: users + songs)
arousal_valence_dict = {}
for _, r in user_mood[["user_id","valence","arousal"]].iterrows():
    arousal_valence_dict[r["user_id"]] = {"valence": r["valence"], "arousal": r["arousal"]}
for _, r in song_data[["song_id","valence","arousal"]].iterrows():
    arousal_valence_dict[r["song_id"]] = {"valence": r["valence"], "arousal": r["arousal"]}

print(f"arousal_valence_dict entries: {len(arousal_valence_dict):,}")
print("Sample user mood:")
display(user_mood[["user_id","arousal","valence"]].head(5))

# ── Mood distribution visualisation ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("User Mood Distribution", fontweight="bold")

axes[0].hist(user_mood["arousal"].dropna(), bins=40, color="#dd8452", edgecolor="white")
axes[0].set_title("Arousal"); axes[0].set_xlabel("Arousal"); axes[0].set_ylabel("Users")

axes[1].hist(user_mood["valence"].dropna(), bins=40, color="#4c72b0", edgecolor="white")
axes[1].set_title("Valence"); axes[1].set_xlabel("Valence")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/mood_dist_{RUN_LABEL}.png", dpi=120)
plt.show()

## §8 · Content Embeddings (BERT on Lyrics — Optional)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §8  BERT Content Embeddings (optional — lyrics required)           ║
# ║  If lyrics_text is empty/missing, we fall back to random projections║
# ║  of the context embeddings so the pipeline remains runnable.        ║
# ╚══════════════════════════════════════════════════════════════════════╝

CONTENT_DIM = 25      # paper uses 25 PCA components from 768-d BERT

# Check lyrics availability
# Convert lyrics_text to string, replace 'nan' with empty string, then check for non-empty
lyrics_available = (song_data["lyrics_text"].astype(str).replace('nan', '').str.strip() != "").sum()
print(f"Songs with lyrics: {lyrics_available:,} / {len(song_data):,}")

RUN_BERT = RUN_BERT_OVERRIDE and lyrics_available > 100

if RUN_BERT:
    # ── Full BERT path ────────────────────────────────────────────────────────
    import torch
    from transformers import BertTokenizer, BertModel

    print("Loading BERT…")
    tokenizer  = BertTokenizer.from_pretrained("bert-base-uncased")
    bert_model = BertModel.from_pretrained("bert-base-uncased",
                                           output_hidden_states=True)
    bert_model.eval().to(DEVICE)

    sentence_embeddings = {}
    BERT_BATCH = 32

    songs_with_lyrics = song_data[
        (song_data["lyrics_text"].astype(str).replace('nan', '').str.strip() != "")
    ][["song_id","lyrics_text"]].reset_index(drop=True)

    for start in tqdm(range(0, len(songs_with_lyrics), BERT_BATCH), desc="BERT"):
        batch_df = songs_with_lyrics.iloc[start:start+BERT_BATCH]
        texts    = [str(t).replace(r"\n"," ")[:512]
                    for t in batch_df["lyrics_text"]]
        enc = tokenizer(texts, padding=True, truncation=True,
                        max_length=128, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            out = bert_model(**enc)
            # Mean of penultimate hidden layer (notebook cell 47 strategy)
            hidden = out.hidden_states[-2]            # (B, T, 768)
            vecs   = hidden.mean(dim=1).cpu().numpy() # (B, 768)
        for sid, vec in zip(batch_df["song_id"], vecs):
            sentence_embeddings[sid] = vec

    # PCA to CONTENT_DIM
    sids  = list(sentence_embeddings.keys())
    mats  = np.stack([sentence_embeddings[s] for s in sids])
    pca_c = PCA(n_components=CONTENT_DIM, random_state=RANDOM_SEED)
    mats_r = pca_c.fit_transform(mats)
    content_embeddings = dict(zip(sids, mats_r))

    # Fill songs without lyrics with zero vector
    zero = np.zeros(CONTENT_DIM, dtype="float32")
    for sid in song_data["song_id"]:
        content_embeddings.setdefault(sid, zero.copy())

    print(f"BERT content_embeddings: {len(content_embeddings):,} songs")
    print(f"PCA variance explained : {pca_c.explained_variance_ratio_.sum()*100:.1f}%")

else:
    # ── Fallback: project context embeddings down to CONTENT_DIM with PCA ─────
    print("Lyrics unavailable/skipped — using PCA projection of context embeddings.")
    song_ids_with_ctx = [sid for sid in song_data["song_id"]
                         if sid in context_embeddings]
    ctx_mat = np.array([context_embeddings[sid] for sid in song_ids_with_ctx],
                       dtype="float32")
    pca_fb  = PCA(n_components=min(CONTENT_DIM, ctx_mat.shape[1]),
                  random_state=RANDOM_SEED)
    proj    = pca_fb.fit_transform(ctx_mat)
    content_embeddings = dict(zip(song_ids_with_ctx, proj))
    zero_c = np.zeros(pca_fb.n_components_, dtype="float32")
    for sid in song_data["song_id"]:
        content_embeddings.setdefault(sid, zero_c.copy())
    CONTENT_DIM = pca_fb.n_components_
    print(f"Fallback CONTENT_DIM   : {CONTENT_DIM}")

print(f"content_embeddings size: {len(content_embeddings):,}")


## §9 · Feature Concatenation & Z-Score Normalisation

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §9  Feature Concatenation                                          ║
# ║  Final embedding = [context(200) | content(25) | arousal | valence] ║
# ║  Then z-score per feature dimension (paper §III-C)                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

def build_embedding(node_id):
    """Concatenate context + content + [arousal, valence]."""
    ctx = context_embeddings.get(node_id)
    cnt = content_embeddings.get(node_id)
    av  = arousal_valence_dict.get(node_id, {"arousal": 0.5, "valence": 0.5})

    # Initialize components with zeros or default values
    ctx_vec = np.zeros(EMBED_DIM, dtype="float32")
    cnt_vec = np.zeros(CONTENT_DIM, dtype="float32")
    av_vec  = np.array([0.5, 0.5], dtype="float32") # Default arousal/valence

    # Assign actual values if available
    if ctx is not None:
        ctx_vec = ctx
    if cnt is not None:
        cnt_vec = cnt
    if av is not None:
        current_arousal = av.get("arousal", 0.5)
        current_valence = av.get("valence", 0.5)
        av_vec = np.array([current_arousal, current_valence], dtype="float32")

    # Ensure all components are finite before concatenation
    # Replace any non-finite values (NaN, inf) with 0.0
    ctx_vec[~np.isfinite(ctx_vec)] = 0.0
    cnt_vec[~np.isfinite(cnt_vec)] = 0.0
    av_vec[~np.isfinite(av_vec)] = 0.0

    return np.concatenate([ctx_vec, cnt_vec, av_vec]).astype("float32")

TOTAL_DIM = EMBED_DIM + CONTENT_DIM + 2
print(f"Total embedding dimension: {TOTAL_DIM}  "
      f"= {EMBED_DIM}(ctx) + {CONTENT_DIM}(cnt) + 2(av)")

# ── Song embeddings ───────────────────────────────────────────────────────────
print("Building song embeddings…")
song_emb_dict = {}
for sid in tqdm(song_data["song_id"], desc="Songs"):
    song_emb_dict[sid] = build_embedding(sid)

# ── User embeddings ───────────────────────────────────────────────────────────
print("Building user embeddings…")
user_emb_dict = {}
for uid in tqdm(user_hyperedges["user_id"], desc="Users"):
    user_emb_dict[uid] = build_embedding(uid)

print(f"  Song embeddings: {len(song_emb_dict):,}")
print(f"  User embeddings: {len(user_emb_dict):,}")

# ── Z-score normalisation (per feature across all entities) ───────────────────
print("Z-score normalisation…")
s_keys, s_vals = zip(*song_emb_dict.items())
u_keys, u_vals = zip(*user_emb_dict.items())

s_mat  = zscore(np.stack(s_vals), axis=0, ddof=1).astype("float32")
u_mat  = zscore(np.stack(u_vals), axis=0, ddof=1).astype("float32")

# Replace NaNs introduced by zscore (e.g., for columns with zero variance)
s_mat[np.isnan(s_mat)] = 0.0
u_mat[np.isnan(u_mat)] = 0.0

song_embeddings = dict(zip(s_keys, s_mat))
user_embeddings = dict(zip(u_keys, u_mat))

print(f"  Song matrix shape: {s_mat.shape}")
print(f"  User matrix shape: {u_mat.shape}")

# ── Feature norms distribution visualisation ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Embedding Norms after Z-score", fontweight="bold")

axes[0].hist(np.linalg.norm(s_mat, axis=1), bins=40, color="#55a868", edgecolor="white")
axes[0].set_title("Song Embedding Norms"); axes[0].set_xlabel("L2 norm")

axes[1].hist(np.linalg.norm(u_mat, axis=1), bins=40, color="#c44e52", edgecolor="white")
axes[1].set_title("User Embedding Norms"); axes[1].set_xlabel("L2 norm")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/embed_norms_{RUN_LABEL}.png", dpi=120)
plt.show()

### §9b · CSV Export — Song Embeddings + Metadata

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §9b  CSV Export — Song Embeddings + Metadata                       ║
# ║  Columns: song_id | artist_name | title | tags | arousal | valence  ║
# ║           | pc1 | pc2 | embed_norm                                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("Building song embedding CSV…")

# 2-D PCA coordinates (for scatter plot and CSV)
_song_ids_csv = [sid for sid in song_data["song_id"] if sid in song_embeddings]
_song_vecs    = np.array([song_embeddings[sid] for sid in _song_ids_csv], dtype="float32")

_pca_csv = PCA(n_components=2, random_state=RANDOM_SEED)
_song_2d = _pca_csv.fit_transform(_song_vecs)

# Helper: extract first tag from stringified list
def first_tag(tag_str):
    try:
        parsed = eval(str(tag_str))
        return parsed[0] if parsed else "unknown"
    except Exception:
        return "unknown"

# Build export DataFrame
_meta = song_data[song_data["song_id"].isin(_song_ids_csv)].set_index("song_id")
_export_rows = []
for i, sid in enumerate(_song_ids_csv):
    row = {
        "song_id"     : sid,
        "artist_name" : _meta.at[sid, "artist_name"] if sid in _meta.index else "",
        "title"       : _meta.at[sid, "title"]       if sid in _meta.index else "",
        "primary_tag" : first_tag(_meta.at[sid, "tags"] if sid in _meta.index else "[]"),
        "arousal"     : float(_meta.at[sid, "arousal"]) if sid in _meta.index else 0.5,
        "valence"     : float(_meta.at[sid, "valence"]) if sid in _meta.index else 0.5,
        "pc1"         : float(_song_2d[i, 0]),
        "pc2"         : float(_song_2d[i, 1]),
        "embed_norm"  : float(np.linalg.norm(_song_vecs[i])),
    }
    _export_rows.append(row)

song_embed_csv = pd.DataFrame(_export_rows)

CSV_PATH = f"{OUT_DIR}/song_embeddings_{RUN_LABEL}.csv"
song_embed_csv.to_csv(CSV_PATH, index=False)
print(f"  CSV saved : {CSV_PATH}")
print(f"  Rows      : {len(song_embed_csv):,}")
print(f"  Columns   : {song_embed_csv.columns.tolist()}")
print(f"  PCA variance explained: {_pca_csv.explained_variance_ratio_.sum()*100:.1f}%")
display(song_embed_csv.head(5))


### §9c · Song Embedding Scatter Plot — Replicating Paper Fig. 3

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §9c  PCA Scatter Plot — Replicating Fig. 3 of La Gatta et al.     ║
# ║  Paper: "Music Recommendation via Hypergraph Embedding"             ║
# ║  IEEE TNNLS 2023  (Fig. 3, p. 7894)                                ║
# ║                                                                     ║
# ║  X-axis : PC1 (first principal component of song embeddings)        ║
# ║  Y-axis : PC2 (second principal component of song embeddings)       ║
# ║  Colour : Primary genre tag assigned by Last.fm                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

try:
    # Use the CSV just exported (ensures CSV and plot are consistent)
    _plot_df = song_embed_csv.copy()

    # Select top tags by frequency (paper shows ~5 genre clusters)
    TOP_N_TAGS = 8
    top_tags   = _plot_df["primary_tag"].value_counts().head(TOP_N_TAGS).index.tolist()
    palette    = dict(zip(top_tags, sns.color_palette("tab10", TOP_N_TAGS)))

    # Split into labelled (top tags) and unlabelled (everything else)
    _plot_df["color_tag"] = _plot_df["primary_tag"].apply(
        lambda t: t if t in top_tags else "_other")
    df_labelled = _plot_df[_plot_df["color_tag"] != "_other"]
    df_other    = _plot_df[_plot_df["color_tag"] == "_other"]

    # ── Figure: mirrors paper Fig. 3 style ────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 7))

    # Background (unlabelled) songs in light grey
    ax.scatter(df_other["pc1"], df_other["pc2"],
               c="#cccccc", s=6, alpha=0.3, linewidths=0, label=None)

    # Foreground: top-tag songs coloured by genre
    for tag in top_tags:
        sub = df_labelled[df_labelled["color_tag"] == tag]
        ax.scatter(sub["pc1"], sub["pc2"],
                   c=[palette[tag]], s=10, alpha=0.65, linewidths=0, label=tag)

    # ── Styling to match paper Fig. 3 ─────────────────────────────────────────
    ax.set_xlabel("Principal Component 1", fontsize=12)
    ax.set_ylabel("Principal Component 2", fontsize=12)
    ax.set_title(
        "2-D PCA Projection of Song Embeddings\n"
        "(Replicating Fig. 3 — La Gatta et al., IEEE TNNLS 2023)",
        fontsize=13, fontweight="bold"
    )
    ax.legend(title="Primary Genre Tag", bbox_to_anchor=(1.01, 1),
              loc="upper left", fontsize=9, framealpha=0.9)
    ax.grid(alpha=0.15)
    sns.despine(ax=ax)

    plt.tight_layout()
    SCATTER_PATH = f"{OUT_DIR}/song_pca_scatter_{RUN_LABEL}.png"
    plt.savefig(SCATTER_PATH, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Scatter plot saved: {SCATTER_PATH}")
    print(f"  Total points   : {len(_plot_df):,}")
    print(f"  Labelled points: {len(df_labelled):,}  ({TOP_N_TAGS} genres)")
    print(f"  Unlabelled     : {len(df_other):,}")

except Exception as e:
    print(f"[PLOT ERROR] Could not generate scatter plot: {e}")
    import traceback; traceback.print_exc()


## §10 · Save Artefacts

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §10  Save artefacts to Google Drive                                ║
# ╚══════════════════════════════════════════════════════════════════════╝

def save_pkl(obj, name):
    path = f"{OUT_DIR}/{name}_{RUN_LABEL}.pkl"
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  Saved: {path}")

save_pkl(hyperedges,         "hyperedges")
save_pkl(vertexMemberships,  "vertexMemberships")
save_pkl(context_embeddings, "context_embeddings")
save_pkl(content_embeddings, "content_embeddings")
save_pkl(arousal_valence_dict, "arousal_valence_dict")
save_pkl(song_embeddings,    "song_embeddings")
save_pkl(user_embeddings,    "user_embeddings")
save_pkl(user_hyperedges,    "user_hyperedges")
w2v_model.save(f"{OUT_DIR}/w2v_model_{RUN_LABEL}.bin")
print("All artefacts saved.")

## §11 · Vectorised Recommendation (GPU-accelerated)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §11  Vectorised Recommendation Generation                          ║
# ║  Implements the appendix formula:                                   ║
# ║    UIscores = (Um * w @ Im.T) / (||Um||_w @ ||Im||_w.T)            ║
# ║  Batched over users to stay within GPU VRAM.                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── Weight vectors (paper §IV, cell 62 of original notebook) ──────────────────
w_context = np.concatenate([np.ones(EMBED_DIM),
                             np.zeros(CONTENT_DIM),
                             np.zeros(2)], dtype="float32")
w_content = np.concatenate([np.zeros(EMBED_DIM),
                             np.ones(CONTENT_DIM),
                             np.zeros(2)], dtype="float32")
w_av      = np.concatenate([np.zeros(EMBED_DIM),
                             np.zeros(CONTENT_DIM),
                             np.ones(2)], dtype="float32")
w_equi    = np.ones(TOTAL_DIM, dtype="float32") / TOTAL_DIM

WEIGHT_MODE = "context"     # ← change: "context" | "content" | "av" | "equi"
W = {"context": w_context, "content": w_content,
     "av": w_av, "equi": w_equi}[WEIGHT_MODE]
print(f"Weight mode: {WEIGHT_MODE}")

# ── Align listened songs and build index ───────────────────────────────────────
listened_song_ids = user_data_total["song_id"].unique()
song_data_listened = (song_data[song_data["song_id"].isin(listened_song_ids)]
                      .reset_index(drop=True))
song_data_listened["matrix_id"] = song_data_listened.index

sid_to_mid = song_data_listened.set_index("song_id")["matrix_id"].to_dict()
user_data_total["song_matrix_id"] = user_data_total["song_id"].map(sid_to_mid)

# ── Test/train split matrices ──────────────────────────────────────────────────
import scipy.sparse as sp

user_test     = user_data_total[user_data_total["set"] == "test"].copy()
user_train_all = user_data_total[user_data_total["set"] == "train"].copy()

user_test = user_test.merge(
    user_hyperedges[["user","user_id","user_id_matrix"]], on="user", how="left")
user_train_all = user_train_all.merge(
    user_hyperedges[["user","user_id","user_id_matrix"]], on="user", how="left")

# Group by user_id_matrix
n_songs = len(song_data_listened)

test_groups  = user_test.groupby("user_id_matrix")["song_matrix_id"].apply(list)
train_groups = (user_train_all[user_train_all["user_id_matrix"]
                               .isin(test_groups.index)]
                .groupby("user_id_matrix")["song_matrix_id"].apply(list))

shared_users = test_groups.index.intersection(train_groups.index)
test_groups  = test_groups.loc[shared_users]
train_groups = train_groups.loc[shared_users]
n_test_users = len(shared_users)
print(f"Evaluation users: {n_test_users:,}  | Songs: {n_songs:,}")

# Build sparse matrices
def build_sparse(groups, n_u, n_s):
    rows, cols = [], []
    for ui, (um, songs) in enumerate(groups.items()):
        for s in songs:
            if pd.notna(s):
                rows.append(ui); cols.append(int(s))
    return sp.csr_matrix(
        (np.ones(len(rows)), (rows, cols)), shape=(n_u, n_s), dtype=np.float32)

users_test_matrix  = build_sparse(test_groups,  n_test_users, n_songs)
users_train_matrix = build_sparse(train_groups, n_test_users, n_songs)
print(f"Test  matrix : {users_test_matrix.shape}  nnz={users_test_matrix.nnz:,}")
print(f"Train matrix : {users_train_matrix.shape}  nnz={users_train_matrix.nnz:,}")

In [ ]:
# ── Build embedding arrays ─────────────────────────────────────────────────────
uid_order = list(shared_users)

# Map user_id_matrix → user_id string
um_to_uid = user_hyperedges.set_index("user_id_matrix")["user_id"].to_dict()

user_emb_arr = np.array(
    [user_embeddings.get(um_to_uid.get(um), np.zeros(TOTAL_DIM))
     for um in uid_order], dtype="float32")

song_emb_arr = np.array(
    [song_embeddings.get(sid, np.zeros(TOTAL_DIM))
     for sid in song_data_listened["song_id"]], dtype="float32")

print(f"user_emb_arr shape : {user_emb_arr.shape}")
print(f"song_emb_arr shape : {song_emb_arr.shape}")

In [ ]:
# ── GPU-accelerated vectorised cosine scoring ──────────────────────────────────
def compute_score_matrix_gpu(U, S, w):
    """
    Weighted cosine similarity matrix between users U and songs S.
    Uses PyTorch on GPU (or falls back to numpy on CPU).
    """
    if DEVICE == "cuda":
        U_t = torch.from_numpy(U).to(DEVICE)
        S_t = torch.from_numpy(S).to(DEVICE)
        w_t = torch.from_numpy(w).to(DEVICE)
        Uw  = U_t * w_t
        num = Uw @ S_t.T
        nU  = (Uw * U_t).sum(1, keepdim=True).sqrt()
        nS  = ((S_t * w_t) * S_t).sum(1, keepdim=True).sqrt()
        den = nU @ nS.T
        scores = (num / den.clamp(min=1e-8)).cpu().numpy()
        del U_t, S_t, w_t, Uw, num, nU, nS, den
        torch.cuda.empty_cache()
    else:
        Uw  = U * w
        num = Uw @ S.T
        nU  = np.sqrt((Uw * U).sum(1, keepdims=True))
        nS  = np.sqrt(((S * w) * S).sum(1, keepdims=True))
        den = nU @ nS.T
        scores = num / np.maximum(den, 1e-8)
    return scores

TOP_K = max(EVAL_K_LIST)
print(f"Scoring {n_test_users:,} users × {n_songs:,} songs (batch={SCORE_BATCH_SIZE})…")
t0 = time.time()

users_predictions = np.zeros((n_test_users, TOP_K), dtype=np.int32)

for i in tqdm(range(0, n_test_users, SCORE_BATCH_SIZE), desc="Scoring batches"):
    j = min(i + SCORE_BATCH_SIZE, n_test_users)
    scores = compute_score_matrix_gpu(user_emb_arr[i:j], song_emb_arr, W)

    # Penalise training songs without dense toarray() expansion
    train_batch = users_train_matrix[i:j]
    for r in range(train_batch.shape[0]):
        start, end = train_batch.indptr[r], train_batch.indptr[r + 1]
        cols = train_batch.indices[start:end]
        if len(cols):
            scores[r, cols] = -3.0

    # Faster than full argsort on every row
    part = np.argpartition(-scores, kth=TOP_K-1, axis=1)[:, :TOP_K]
    row_idx = np.arange(part.shape[0])[:, None]
    part_scores = scores[row_idx, part]
    order = np.argsort(-part_scores, axis=1)
    users_predictions[i:j] = part[row_idx, order]

    del scores, train_batch, part, part_scores, order
    gc.collect()

print(f"Scoring done in {time.time()-t0:.1f}s")


## §12 · Evaluation — Recall@K & MAP@K

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §12  Evaluation                                                    ║
# ║  Metrics: Recall@K and Mean Average Precision @K                   ║
# ║  for K ∈ EVAL_K_LIST (paper Table III)                             ║
# ╚══════════════════════════════════════════════════════════════════════╝

def recall_and_ap_at_k(test_mat, predictions, k):
    """Vectorised Recall@K and AP@K over all users."""
    recalls, aps = [], []
    for i in range(test_mat.shape[0]):
        test_songs = set(test_mat[i].indices)
        top_k      = predictions[i, :k].tolist()
        if not test_songs:
            continue

        # Recall
        tp = sum(1 for s in top_k if s in test_songs)
        recalls.append(tp / len(test_songs))

        # Average Precision
        found, score = 0, 0.0
        for rank, s in enumerate(top_k, 1):
            if s in test_songs:
                found += 1
                score += found / rank
        aps.append(score / min(len(test_songs), k))

    return np.mean(recalls), np.mean(aps)

print(f"\n{'─'*55}")
print(f"  {'K':>5}  {'Recall@K':>12}  {'MAP@K':>12}")
print(f"{'─'*55}")

results = {}
for k in EVAL_K_LIST:
    recall, ap = recall_and_ap_at_k(users_test_matrix, users_predictions, k)
    results[k] = (recall, ap)
    print(f"  {k:>5}  {recall*100:>11.2f}%  {ap*100:>11.2f}%")

print(f"{'─'*55}")

In [ ]:
# ── Results visualisation ──────────────────────────────────────────────────────
ks     = list(results.keys())
recalls = [results[k][0]*100 for k in ks]
maps    = [results[k][1]*100 for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"HEMR Recommendation Quality ({RUN_LABEL})", fontweight="bold")

axes[0].plot(ks, recalls, marker="o", color="#4C72B0", linewidth=2)
axes[0].set_title("Recall@K"); axes[0].set_xlabel("K")
axes[0].set_ylabel("Recall (%)"); axes[0].grid(alpha=0.3)
for x, y in zip(ks, recalls):
    axes[0].annotate(f"{y:.1f}%", (x, y), textcoords="offset points",
                     xytext=(0, 6), ha="center", fontsize=9)

axes[1].plot(ks, maps, marker="s", color="#dd8452", linewidth=2)
axes[1].set_title("MAP@K"); axes[1].set_xlabel("K")
axes[1].set_ylabel("MAP (%)"); axes[1].grid(alpha=0.3)
for x, y in zip(ks, maps):
    axes[1].annotate(f"{y:.1f}%", (x, y), textcoords="offset points",
                     xytext=(0, 6), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eval_results_{RUN_LABEL}.png", dpi=120)
plt.show()

## §13 · Cold-Start Experiment

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §13  Cold-Start Experiment (paper §IV-C-3)                        ║
# ║  Evaluate only on test users with ≤5 listened songs in train set.  ║
# ╚══════════════════════════════════════════════════════════════════════╝

COLD_START_THRESHOLD = 5

cold_mask = np.array(
    [users_train_matrix[i].nnz <= COLD_START_THRESHOLD
     for i in range(n_test_users)])

print(f"Cold-start users (≤{COLD_START_THRESHOLD} train songs): "
      f"{cold_mask.sum():,} / {n_test_users:,}")

if cold_mask.sum() > 0:
    cold_test  = users_test_matrix[cold_mask]
    cold_preds = users_predictions[cold_mask]

    print(f"\n{'─'*55}  [COLD START]")
    print(f"  {'K':>5}  {'Recall@K':>12}  {'MAP@K':>12}")
    print(f"{'─'*55}")
    cold_results = {}
    for k in EVAL_K_LIST:
        recall, ap = recall_and_ap_at_k(cold_test, cold_preds, k)
        cold_results[k] = (recall, ap)
        print(f"  {k:>5}  {recall*100:>11.2f}%  {ap*100:>11.2f}%")
    print(f"{'─'*55}")

    # Overlay plot
    ks_c = list(cold_results.keys())
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("All Users vs Cold-Start Users", fontweight="bold")

    for ax, metric, idx, label in [
        (axes[0], "Recall@K", 0, "Recall (%)"),
        (axes[1], "MAP@K",    1, "MAP (%)")
    ]:
        full_vals = [results[k][idx]*100 for k in ks_c]
        cold_vals = [cold_results[k][idx]*100 for k in ks_c]
        ax.plot(ks_c, full_vals, marker="o", label="All users",  color="#4C72B0")
        ax.plot(ks_c, cold_vals, marker="s", label="Cold-start", color="#c44e52",
                linestyle="--")
        ax.set_title(metric); ax.set_xlabel("K"); ax.set_ylabel(label)
        ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/cold_start_{RUN_LABEL}.png", dpi=120)
    plt.show()
else:
    print("Not enough cold-start users in this subset.")

## §14 · Sample Recommendations for a Random Test User

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §14  Sample Recommendations                                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

sample_ui = random.randint(0, n_test_users - 1)
print(f"Sample user index: {sample_ui}")

train_song_ids = set(users_train_matrix[sample_ui].indices)
test_song_ids  = set(users_test_matrix[sample_ui].indices)
top_pred_ids   = users_predictions[sample_ui, :20].tolist()

id_to_sid = song_data_listened["song_id"].to_dict()

print(f"\n  Training songs ({len(train_song_ids)}):")
train_df = song_data_listened[song_data_listened["matrix_id"].isin(train_song_ids)]
display(train_df[["song_id","artist_name","title"]].head(5))

print(f"\n  Test (ground-truth) songs ({len(test_song_ids)}):")
test_df = song_data_listened[song_data_listened["matrix_id"].isin(test_song_ids)]
display(test_df[["song_id","artist_name","title"]].head(5))

print("\n  Top-20 Recommendations:")
rec_df = song_data_listened[song_data_listened["matrix_id"].isin(top_pred_ids)].copy()
rec_df["rank"] = rec_df["matrix_id"].map(
    {mid: rank for rank, mid in enumerate(top_pred_ids, 1)})
rec_df["hit"] = rec_df["matrix_id"].isin(test_song_ids)
display(rec_df.sort_values("rank")[["rank","song_id","artist_name","title","hit"]])

## §15 · Runtime & Memory Summary

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  §15  Runtime & Memory Summary                                      ║
# ╚══════════════════════════════════════════════════════════════════════╝
import psutil

mem          = psutil.virtual_memory()
total_elapsed = time.time() - _NOTEBOOK_START

print("╔══════════════════════════════════════════════════════════╗")
print("║             HEMR Run Summary                            ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Mode          : {RUN_LABEL:<38} ║")
print(f"║  Device        : {DEVICE:<38} ║")
print(f"║  Total time    : {total_elapsed/60:<37.1f}m ║")
print(f"║  Songs         : {len(song_data):<38,} ║")
print(f"║  Users (train) : {user_data['user'].nunique():<38,} ║")
print(f"║  Hyperedges    : {len(hyperedges):<38,} ║")
print(f"║  Vertices      : {len(vertexMemberships):<38,} ║")
print(f"║  Walks         : {len(walksSAT):<38,} ║")
print(f"║  Embed dim     : {TOTAL_DIM:<38} ║")
print(f"║  RAM used      : {mem.used/1e9:<37.1f}G ║")
print(f"║  RAM total     : {mem.total/1e9:<37.1f}G ║")
if DEVICE == "cuda":
    vram_used = torch.cuda.memory_allocated()/1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"║  VRAM used     : {vram_used:<37.1f}G ║")
    print(f"║  VRAM total    : {vram_total:<37.1f}G ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Evaluation Results (Recall@K / MAP@K)                 ║")
for k, (r, ap) in results.items():
    line = f"@{k:<4}  Recall={r*100:5.2f}%   MAP={ap*100:5.2f}%"
    print(f"║    {line:<52} ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  CSV output    : song_embeddings_{RUN_LABEL}.csv{'':<14} ║")
print(f"║  Scatter plot  : song_pca_scatter_{RUN_LABEL}.png{'':<13} ║")
print(f"║  All outputs   : {OUT_DIR:<38} ║")
print("╚══════════════════════════════════════════════════════════╝")
